In [ ]:
import numpy as np

In [ ]:
print(np.__version__)

In [ ]:
#!pip uninstall numpy -y
#!pip install numpy<2 --no-cache-dir

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder,MinMaxScaler,RobustScaler
import warnings
import matplotlib.pyplot as plt

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv("loan_approved.csv")

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

#### Data Cleaning
- Dropping columns: dropna(), drop(), drop_duplicates()
- Filling/Imputing: fillna(), or impute manually
- Winsorization

In [ ]:
df.duplicated().sum()   #To get the number of duplicate rows
df.drop_duplicates()    #To remove or delete or drop duplicate rows from data

In [ ]:
df.shape

In [ ]:
#df.drop('Gender',axis=1,inplace=True)
df.dropna()

In [ ]:
#Null columns: 'Gender','Dependents','Self_Employed','LoanAmount','Loan_Amount_Term','Credit_History'

#df.fillna(value = {'Gender': 'F','LoanAmount':np.mean(df.LoanAmount),'Credit_History':df.Credit_History.median()})
#df.fillna(0)

#df.loc[df.LoanAmount.isnull(),'LoanAmount'] = df.LoanAmount.mean()  # - Each individual column can be imputed separately

In [ ]:
df.Loan_Amount_Term.value_counts()

In [ ]:
df.fillna(value = {'Gender': 'Male','LoanAmount':df.LoanAmount.median(),'Credit_History':df.Credit_History.median(),'Dependents':'0','Self_Employed':'No','Loan_Amount_Term':360.0},inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.Married.value_counts()

In [ ]:
df.loc[df.Married.isnull(),'Married'] = 'Yes'
#df.loc[(df.Married.isnull()&df.Dependents=='0'),'Married']='No'
#df.loc[(df.Married.isnull()&df.Dependents!='0'),'Married']='Yes'
#df.loc[(df.Married.isnull()&df.CoapplicantIncome==0),'Married']='No'
#df.loc[(df.Married.isnull()&df.CoapplicantIncome!=0),'Married']='Yes'

In [ ]:
df.isnull().sum()

### Outlier Handling

In [ ]:
#Winsorization
sns.boxplot(x=df.ApplicantIncome ,orient='h')

In [ ]:
from scipy.stats.mstats import winsorize
#Specify trim percentage
trim_p = 0.08
df['ApplicantIncome1'] = winsorize(df['ApplicantIncome'],limits=trim_p)

In [ ]:
sns.boxplot(x=df.ApplicantIncome1 ,orient='h')

In [ ]:
#Manually setting limits
df.loc[df.ApplicantIncome>20000,'ApplicantIncome'] = 20000

In [ ]:
#Removing Outliers using IQR

Q1 = df['LoanAmount'].quantile(0.25)
Q3 = df['LoanAmount'].quantile(0.75)
IQR = Q3 - Q1

# Define lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Capping outliers
df.loc[(df['LoanAmount'] <= lower_bound),'LoanAmount'] = lower_bound
df.loc[(df['LoanAmount'] >= upper_bound),'LoanAmount'] = upper_bound

In [ ]:
sns.boxplot(x=df.LoanAmount ,orient='h')

## Types Of Transformation

1. Scaling is also indirectly transforming the variable. Standardization, Normal MinMax, Robust scaling are the techniques used.
3. Logarithmic Transformation
4. Exponential Transformation
5. Box Cox Transformation
6. Square Root Transformation

In [ ]:
df['CoapplicantIncome_log'] = np.log(df['CoapplicantIncome'])
plt.figure(figsize=(6,4))
plt.subplot(1,2,1)
sns.histplot(x=df['CoapplicantIncome'], kde= True)
plt.subplot(1,2,2)
sns.histplot(x=df['CoapplicantIncome_log'], kde= True)
plt.tight_layout()

In [ ]:
df['ApplicantIncome_exp1']=df.ApplicantIncome**(0.5)
df['ApplicantIncome_exp2']=df.ApplicantIncome**(0.1)
plt.figure(figsize=(6,3))
plt.subplot(1,3,1)
sns.histplot(x=df['ApplicantIncome'], kde= True)
plt.subplot(1,3,2)
sns.histplot(x=df['ApplicantIncome_exp1'], kde= True)
plt.subplot(1,3,3)
sns.histplot(x=df['ApplicantIncome_exp2'], kde= True)

In [ ]:
from scipy.stats import boxcox
transformed_data, lambda_value = boxcox(df['ApplicantIncome'])

In [ ]:
transformed_data

In [ ]:
lambda_value

In [ ]:
df['ApplicantIncome_boxcox'] = transformed_data
plt.figure(figsize=(4,2))
plt.subplot(1,2,1)
sns.histplot(x=df['ApplicantIncome'], kde= True)
plt.subplot(1,2,2)
sns.histplot(x=df['ApplicantIncome_boxcox'], kde= True)

In [ ]:
#from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder
df

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
ApplicantIncome_scaled = scaler.fit_transform(df[['ApplicantIncome']])

In [ ]:
ApplicantIncome_scaled

In [ ]:
from sklearn.preprocessing import MinMaxScaler
#ApplicantIncome_scaled2 = MinMaxScaler().fit_transform(df[['ApplicantIncome']])
ApplicantIncome_scaled2 = scaler.fit_transform(df[['ApplicantIncome']])

In [ ]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
ApplicantIncome_scaled3 = scaler.fit_transform(df[['ApplicantIncome']])

### Encoding
- Converting categorical variables into numerical so that the model can process them
- OHE (one hot encoding) - Create as many columns as unique values of the categorical feature. Done to those features that cannot(shouldn't) have an ordinality.
- Label Encoding - Let the default lexicographic (alphabetical order) endocing happen. Numbers automatically assigned without any order. Renders a 1-d output (unlike all other scalers and encoders that render a 2d output) as it is designed to encode target columns
- Ordinal Encoding - Speicify the order in which numerical labels must be assigned. Done when the data is ordinal
- Frequency Encoding - Specify the order based on frequency or value_counts

In [ ]:
df.columns

In [ ]:
# OHE - One Hot Encoding
#df.Married.value_counts()
pd.get_dummies(df, columns=['Married'],dtype='int')

In [ ]:
#OHE from sklearn
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
result = enc.fit_transform(df[['Married']]).toarray()

In [ ]:
df.Married.unique()

In [ ]:
result

In [ ]:
#Label Encoder
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
df['Gender_enc'] = enc.fit_transform(df.Gender)
df[['Gender','Gender_enc']]

In [ ]:
enc.fit_transform(df.Gender)

In [ ]:
#Frequency encoding using Ordinal Encoder with value_counts
from sklearn.preprocessing import OrdinalEncoder
df.Gender.value_counts()
enc = OrdinalEncoder(categories=[['Male','Female']])
df['Gender_OE1'] = enc.fit_transform(df[['Gender']])


In [ ]:
#Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder
df.Property_Area.unique()
enc = OrdinalEncoder(categories=[['Rural','Semiurban','Urban']])
df['OE_Prop_Area'] = enc.fit_transform(df[['Property_Area']])
df[['Property_Area','OE_Prop_Area']]

In [ ]:
#Frequency encoding using Ordinal Encoder with value_counts
from sklearn.preprocessing import OrdinalEncoder
df.Gender.value_counts()
enc = OrdinalEncoder(categories=[['Male','Female']])
df['Gender_OE1'] = enc.fit_transform(df[['Gender']])

In [ ]:
df.Gender_OE1

In [ ]:
enc.fit_transform(df[['Gender']])

### Feature Creation, Feature Selection, Feature Extraction/Sythesis, Feature Reduction
- Feature creation: Create new columns based on intuition

### PCA - Principal Component Analysis
- Dimensionality curse: After a certain number of columns, distances between the points will become the same. And in that case, the distance based algorithms won't work. In common words, the data has too many features/dependencies that makes it difficult to fit a model, and we are likely to overfit
- In such cases we try to reduce the dimension of the problem (or columns in data). Many techniques are there, principle component Analysis is one of them.
- Correlation is also used to drop columns with high correlation, but when columns aren't correlated with the output, we gotta drop some. We might want to filter them based on relevance to target variable, and the variability within the column.
- PCA comes into picture when all variables are relevant and have no correlation within them.
- Influence of a column on the target variable is found out from the variance within the column. Higher the variance of data in column, higher is the information it contains about target variable, or higher is its influence on target variable.
- Every column is compared to other, we try to shift axis such that variance along one axis shall be maximum and other axis be minumum. In that sense, we drop the axis with minimum variance, and retain the one with maximum variance which is now a function of original two columns. This is called a principle component.
- PCA Algorithm gives all the principal components in ranking order of variance. Each  Within 5-6 components, we can cover 90% of variance of target. In that case, we drop the remaining principal components
- The principal component analysis is an unsupervised machine learning algorithm used for feature selection using dimensionality reduction techniques. As the name suggests, it finds out the principal components from the data. PCA transforms and fits the data from a higher-dimensional space to a new, lower-dimensional subspace This results into an entirely new coordinate system of the points where the first axis corresponds to the first principal component that explains the most variance in the data.

**What are the principal components?**
Principal components are the derived features which explain the maximum variance in the data. The first principal component explains the most variance, the 2nd a bit less and so on. Each of the new dimensions found using PCA is a linear combination of the old features.

## Scree Plots:
Scree plots are the graphs that convey how much variance is explained by corresponding Principal components.


In [ ]:
# importing important labries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.get_dataset_names()

In [ ]:
data = sns.load_dataset('brain_networks')
pd.reset_option('display.max_rows',None)
pd.reset_option('display.max_columns',None)
data

In [ ]:
data.to_csv('output.csv',index=False)

In [ ]:
data = sns.load_dataset('brain_networks').iloc[3:,1:]

In [ ]:
pd.reset_option('display.max_rows',None)
pd.reset_option('display.max_columns',None)
data

In [ ]:
data.reset_index(drop=True,inplace=True)

In [ ]:
data

In [ ]:
#data.info(),data.describe(),data.head(),data.columns,data.isnull().sum()
pd.set_option('display.max_rows',None)
data.isnull().sum()

In [ ]:
pd.reset_option('display.max_rows',None)

In [ ]:
data.info()

In [ ]:
data = data.astype('float')

In [ ]:
data.info()

In [ ]:
# Applying StandardScaler
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaled_data=scaler.fit_transform(data)
col = data.columns
data = pd.DataFrame(data=scaled_data,columns=col)
pd.reset_option('display.max_rows',None)

In [ ]:
scaled_data

In [ ]:
data

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
PC = pca.fit_transform(data)
PC

In [ ]:
PC.shape

In [ ]:
pca.explained_variance_ratio_

In [ ]:
a  = np.array([1,2,3,3,2,4,5,3,2])
np.cumsum(a)

In [ ]:
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
np.where(np.cumsum(pca.explained_variance_ratio_)>0.9)

In [ ]:
plt.figure(figsize=(5,3))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('No. of PC')
plt.ylabel('Explained variance')
plt.title('Explained Variance by PC - Scree plots')
plt.show()
#plt.tight_layout()

In [ ]:
pca2 = PCA(n_components=32)
data_PCA = pd.DataFrame(pca2.fit_transform(data))

In [ ]:
data_PCA.head()